# পাঠ ০৫ - এজেন্টিক RAG


## সেটআপ

এই নোটবুকটি মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্ক ব্যবহার করে Agentic RAG (Retrieval-Augmented Generation) প্যাটার্ন প্রদর্শন করে।

**প্রয়োজনীয়তাসমূহ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — আপনার Azure AI সার্চ সার্ভিস এন্ডপয়েন্ট
- `AZURE_SEARCH_API_KEY` — আপনার Azure AI সার্চ API কী
- পরিবেশ ভেরিয়েবল মারফত কনফিগার করা Azure OpenAI ডিপ্লয়মেন্ট
- Azure CLI তে প্রমাণীকৃত (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG কী?

প্রথাগত RAG একটি নির্দিষ্ট পদ্ধতি অনুসরণ করে: প্রথমে ডকুমেন্ট পুনরুদ্ধার করা হয়, তারপর একটি উত্তর তৈরি করা হয়। **Agentic RAG** একটি ধাপ এগিয়ে যায় যা এজেন্টকে স্বাধীনতা দেয় সিদ্ধান্ত নিতে **কখন** এবং **কেমনভাবে** তথ্য পুনরুদ্ধার করতে হবে।

Agentic RAG-এর সাথে, এজেন্ট করতে পারে:
- **সিদ্ধান্ত** নেওয়া উত্তর দেওয়ার আগে পুনরুদ্ধারের প্রয়োজন আছে কিনা
- **বাছাই** করা কোন তথ্য উৎস বা সরঞ্জাম থেকে তথ্য সংগ্রহ করতে হবে
- **মূল্যায়ন** করা পুনরুদ্ধার ফলাফল এবং যদি প্রথম প্রচেষ্টা যথেষ্ট না হয় তবে পরবর্তী পুনরুদ্ধার সম্পাদন করা
- **মিশ্রণ** করা একাধিক পুনরুদ্ধার ধাপ থেকে তথ্য একটি সুসংগঠিত উত্তরে

এটি এজেন্টকে একটি স্থির পুনরুদ্ধার-তারপর-উৎপাদন পদ্ধতির তুলনায় আরও নমনীয় এবং সঠিক করে তোলে।


## একটি অনুসন্ধান সরঞ্জাম তৈরি করা

Agentic RAG-এ, বাহ্যিক তথ্য উত্সগুলোকে **সরঞ্জাম** আকারে মোড়ানো হয় যা এজেন্টের প্রয়োজন অনুযায়ী আহ্বান করতে পারে। এটি এজেন্টকে পুনরুদ্ধারকে আরেকটি কর্ম হিসেবে গ্রহণ করতে দেয়, বাধ্যতামূলক ধাপ হিসেবে নয়।

নীচে আমরা একটি ভ্রমণ জ্ঞানভিত্তিক ডাটাবেস সংজ্ঞায়িত করি এবং এটিকে একটি সরঞ্জাম হিসেবে প্রকাশ করি যা এজেন্ট গন্তব্য তথ্য খোঁজার জন্য কল করতে পারে।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG এজেন্ট তৈরি করা

এখন আমরা একটি এজেন্ট তৈরি করব যাকে নির্দেশ দেওয়া হয়েছে **উত্তর দেওয়ার আগে সবসময় তথ্য উদ্ধার করতে**। এজেন্টটি তার নিজস্ব প্রশিক্ষণ ডেটার উপর নির্ভর না করে জ্ঞানভিত্তিক উত্তরের জন্য `search_travel_knowledge` টুলটি ব্যবহার করে।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## পুনরাবৃত্ত অনুসন্ধান — মেকার-চেকার প্যাটার্ন

Agentic RAG এর একটি প্রধান সুবিধা হল **পুনরাবৃত্ত অনুসন্ধান**। এজেন্ট একাধিক বার অনুসন্ধান করে তার প্রাথমিক তথ্য যাচাই, পরিমার্জন বা বিস্তারের কাজ করতে পারে — যা একটি "মেকার-চেকার" ওয়ার্কফ্লোর মতো:

1. **মেকার ধাপ**: এজেন্ট প্রাথমিক তথ্য অনুসন্ধান করে এবং একটি উত্তর খসড়া প্রস্তুত করে।
2. **চেকার ধাপ**: এজেন্ট অতিরিক্ত অনুসন্ধান করে বিশদ যাচাই বা ফাঁক পূরণ করে।

নিম্নে, এজেন্টকে এমন একটি প্রশ্ন দেওয়া হয়েছে যা একাধিক গন্তব্যের তুলনা করতে হয়, ফলে এটি একাধিকবার অনুসন্ধান করতে বাধ্য হয়।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## সারসংক্ষেপ

এই পাঠে আপনি শিখেছেন কিভাবে Microsoft Agent Framework ব্যবহার করে একটি **Agentic RAG** সিস্টেম তৈরি করা যায়:

- **Agentic RAG** এজেন্টদের স্বয়ংক্রিয়ভাবে সিদ্ধান্ত নেওয়ার সুযোগ দেয় কখন তথ্য সংগ্রহ করতে হবে, যার ফলে সংগ্রহ গতিশীল হয়, নির্দিষ্ট নয়।
- **টুলস হিসাবে তথ্য উৎস**: বাইরের জ্ঞানভান্ডার (যেমন Azure AI Search) টুলস হিসেবে মোড়ানো হয় যেগুলো এজেন্ট কল করতে পারে।
- **পুনরাবৃত্তি সংগ্রহ**: মেকার-চেকার প্যাটার্ন এজেন্টকে একাধিকবার তথ্য অনুসন্ধান, যাচাই এবং পরিমার্জন করার সুযোগ দেয় — চূড়ান্ত উত্তর দেওয়ার আগে।

প্রোডাকশনে, বড় পরিমাণে ট্রাভেল ডকুমেন্ট সংগ্রহের জন্য আপনি ইন-মেমরি `TRAVEL_KNOWLEDGE_BASE` এর পরিবর্তে প্রকৃত Azure AI Search সূচক ব্যবহার করবেন।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা যথাসাধ্য সঠিকতার চেষ্টা করি, অনুগ্রহ করে লক্ষ্য করুন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথির স্থানীয় ভাষার সংস্করণই প্রধান এবং বিশ্বাসযোগ্য উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানুষের অনুবাদ গ্রহণ করাই বাঞ্ছনীয়। এই অনুবাদের ব্যবহার থেকে সৃষ্ট কোনো ভুলবোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
